In [3]:
import json
import pandas as pd
from pathlib import Path

# Download and save raw data in the usual format

In [2]:
!wget https://raw.githubusercontent.com/Unbabel/BConTrasT/refs/heads/master/test1_corpus-with_gold_references.json -O ../downloaded_data/bcontrast_test.json

--2024-10-23 18:22:13--  https://raw.githubusercontent.com/Unbabel/BConTrasT/refs/heads/master/test1_corpus-with_gold_references.json


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 478274 (467K) [text/plain]
Saving to: ‘../downloaded_data/bcontrast_test.json’

../downloaded_data/ 100%[===================>] 467.06K  --.-KB/s    in 0.07s   

2024-10-23 18:22:13 (6.33 MB/s) - ‘../downloaded_data/bcontrast_test.json’ saved [478274/478274]



### Create long df for conversations and lp dfs

In [5]:
orig_json_path = "../downloaded_data/bcontrast_test.json"
data = json.load(open(orig_json_path))
long_data_dict = {
    "source_language": [],
    "target_language": [],
    "src": [],
    "ref": [],
    "doc_id": [],
    "client_id": [],
    "sender": [],
}
for convo in data.keys():
    for turn in data[convo]:
        speaker = turn["speaker"]
        if speaker == "agent":
            long_data_dict["source_language"].append("en")
            long_data_dict["target_language"].append("de")
        else:
            long_data_dict["source_language"].append("de")
            long_data_dict["target_language"].append("en")
        long_data_dict["src"].append(turn["source"])
        long_data_dict["ref"].append(turn["target"])
        long_data_dict["doc_id"].append(convo)
        long_data_dict["client_id"].append(convo)
        long_data_dict["sender"].append(turn["speaker"])

long_df = pd.DataFrame(long_data_dict)

en_de_df = long_df[long_df["source_language"] == "en"].reset_index(drop=True)
de_en_df = long_df[long_df["source_language"] == "de"].reset_index(drop=True)

paths = [
    "../raw_data/mt/bcontrast_test.de",
    "../raw_data/mt/bcontrast_test.en-de",
    "../raw_data/mt/bcontrast_test.de-en",
]

for path, df in zip(paths, [long_df, en_de_df, de_en_df]):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    df.to_json(path / "test.jsonl", lines=True, orient="records", force_ascii=False)

In [4]:
long_df.head()

,source_language,target_language,src,ref,doc_id,client_id,sender
0,en,de,"hello, how can i help you?","Hallo, wie kann ich Ihnen helfen?",dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,agent
1,de,en,Hallo. Ich versuche Kinokarten im AMC Houston ...,Hi. I'm trying to buy movie tickets at the AMC...,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,customer
2,de,en,"Ich hatte gehofft, Sie könnten mir helfen, die...",I was hoping you could help me place that order.,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,customer
3,en,de,"sure, i can help you with that for sure. what ...","Natürlich, ich kann Ihnen gerne dabei helfen. ...",dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,agent
4,de,en,Glass.,glass.,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,dlg-0093d9ca-b42d-4b3c-9a89-c872000915f6,customer


# Create instructions for TowerInstruct and TowerChat, full and no context

In [6]:
from transformers import AutoTokenizer
from tower_eval.utils import write_lines

dataset_name = "bcontrast_test"
raw_data_dir = Path("../raw_data/")

code_lang_dict = {
    "en": "English",
    "de": "German",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

In [3]:
def build_no_context(lps, instructions_data_dir, code_lang_dict, apply_instruction, t):
    for lp in lps:
        print(lp)
        if lp == "en-pt-br":
            non_eng = "pt-br"
            eng = "en"
        else:
            eng, non_eng = lp.split("-")

        # en-xx direction
        save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
        df = pd.read_json(
            raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
        )
        instructions = (
            df["src"]
            .apply(
                lambda x: apply_instruction(
                    x, code_lang_dict[eng], code_lang_dict[non_eng], t
                )
            )
            .tolist()
        )
        write_lines(
            save_dir / "instructions.txt",
            instructions,
            escape_newline=True,
            verbose=False,
        )
        # xx-en direction
        save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
        df = pd.read_json(
            raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
        )
        instructions = (
            df["src"]
            .apply(
                lambda x: apply_instruction(
                    x, code_lang_dict[non_eng], code_lang_dict[eng], t
                )
            )
            .tolist()
        )
        write_lines(
            save_dir / "instructions.txt",
            instructions,
            escape_newline=True,
            verbose=False,
        )


def build_full_context(
    lps,
    instructions_data_dir,
    code_lang_dict,
    apply_instruction,
    apply_series_of_instruction,
    t,
):
    for lp in lps:
        print(lp)
        if lp == "en-pt-br":
            non_eng = "pt-br"
            eng = "en"
        else:
            eng, non_eng = lp.split("-")

        instructions = []
        df = pd.read_json(
            f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
            lines=True,
        )
        register_convo = False
        last_i = 0
        for row_i, row in df.iterrows():
            convo_doc_id = row["doc_id"]
            if row_i != len(df) - 1:
                next_doc_id = df.loc[row_i + 1, "doc_id"]
                if convo_doc_id != next_doc_id:
                    register_convo = True
                    convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                    last_i = row_i + 1
            else:
                register_convo = True
                convo_df = df.loc[last_i:].reset_index(drop=True)
            if register_convo:
                for i, row in convo_df.iterrows():
                    src_lang = code_lang_dict[row["source_language"]]
                    tgt_lang = code_lang_dict[row["target_language"]]
                    if i == 0:
                        instructions.append(
                            apply_instruction(
                                row["src"],
                                src_lang,
                                tgt_lang,
                                t,
                            )
                        )
                    else:
                        previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                        previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                        instructions.append(
                            apply_series_of_instruction(
                                row["src"],
                                src_lang,
                                tgt_lang,
                                t,
                                previous_sources=previous_sources,
                                previous_directions=previous_directions,
                                current_direction=row["sender"],
                            )
                        )
            register_convo = False

        df["instructions"] = instructions
        # en-xx direction
        save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
        instructions = df[df["source_language"] == "en"]["instructions"].tolist()
        write_lines(
            save_dir / "instructions.txt",
            instructions,
            escape_newline=True,
            verbose=False,
        )
        # xx-en direction
        save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
        instructions = df[df["target_language"] == "en"]["instructions"].tolist()
        write_lines(
            save_dir / "instructions.txt",
            instructions,
            escape_newline=True,
            verbose=False,
        )

## TowerInstruct

In [38]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")
    messages = [
        {
            "role": "user",
            "content": final_string,
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


t = AutoTokenizer.from_pretrained("Unbabel/TowerInstruct-13B-v0.2")

instructions_data_dir = Path("../instructions/no_context")
build_no_context(lps, instructions_data_dir, code_lang_dict, apply_instruction, t)
instructions_data_dir = Path("../instructions/full_context")
build_full_context(
    lps,
    instructions_data_dir,
    code_lang_dict,
    apply_instruction,
    apply_series_of_instruction,
    t,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


en-de
en-de


# TowerChat

In [39]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "system",
            "content": "",
        },
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")
    messages = [
        {
            "role": "system",
            "content": "",
        },
        {
            "role": "user",
            "content": final_string,
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


t = AutoTokenizer.from_pretrained("Unbabel/TowerInstruct-v0.2-w-chat-mt-data")

instructions_data_dir = Path("../instructions/no_context_empty_sys")
build_no_context(lps, instructions_data_dir, code_lang_dict, apply_instruction, t)
instructions_data_dir = Path("../instructions/full_context_empty_sys")
build_full_context(
    lps,
    instructions_data_dir,
    code_lang_dict,
    apply_instruction,
    apply_series_of_instruction,
    t,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


en-de
en-de


# Open AI

In [4]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    return f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: "


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return final_string


t = None

instructions_data_dir = Path("../instructions/no_context_no_template")
build_no_context(lps, instructions_data_dir, code_lang_dict, apply_instruction, t)
instructions_data_dir = Path("../instructions/full_context_no_template")
build_full_context(
    lps,
    instructions_data_dir,
    code_lang_dict,
    apply_instruction,
    apply_series_of_instruction,
    t,
)

en-de
en-de


In [5]:
lps

['en-de']